&nbsp;
## 4.6 编写 GPT 模型代码

[返回 4.1–4.5 节](ch04.ipynb) | [前往 4.7 节](ch04_4.7.ipynb)

**独立运行准备：**下面的单元从 `gpt.py` 导入前面实现的 `LayerNorm` 和 `TransformerBlock`，并重建 GPT-2 配置和示例输入。修改前面笔记本中的组件不会自动更改 `gpt.py`。

In [1]:
import torch
import torch.nn as nn
import tiktoken
from gpt import LayerNorm, TransformerBlock

GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

tokenizer = tiktoken.get_encoding("gpt2")
batch = torch.stack([
    torch.tensor(tokenizer.encode("Every effort moves you")),
    torch.tensor(tokenizer.encode("Every day holds a")),
], dim=0)

- 我们即将大功告成：现在我们将 Transformer 块插入到本章开头编写的架构中，从而构建一个可用的完整 GPT 架构
- 请注意，Transformer 块会被多次重复堆叠；在最小的 124M GPT-2 模型中，我们将其重复 12 次：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/15.webp" width="600px">

- 对应的代码实现如下，其中 `cfg["n_layers"] = 12`：

In [2]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

- 使用 124M 参数模型的配置，我们现在可以用随机初始权重实例化该 GPT 模型，如下所示：

In [3]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)

out = model(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

Input batch:
 tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

Output shape: torch.Size([2, 4, 50257])
tensor([[[ 0.3613,  0.4222, -0.0711,  ...,  0.3483,  0.4661, -0.2838],
         [-0.1792, -0.5660, -0.9485,  ...,  0.0477,  0.5181, -0.3168],
         [ 0.7120,  0.0332,  0.1085,  ...,  0.1018, -0.4327, -0.2553],
         [-1.0076,  0.3418, -0.1190,  ...,  0.7195,  0.4023,  0.0532]],

        [[-0.2564,  0.0900,  0.0335,  ...,  0.2659,  0.4454, -0.6806],
         [ 0.1230,  0.3653, -0.2074,  ...,  0.7705,  0.2710,  0.2246],
         [ 1.0558,  1.0318, -0.2800,  ...,  0.6936,  0.3205, -0.3178],
         [-0.1565,  0.3926,  0.3288,  ...,  1.2630, -0.1858,  0.0388]]],
       grad_fn=<UnsafeViewBackward0>)


- 我们将在下一章中训练这个模型
- 不过，关于它的参数量有一点需要简要说明：我们之前将其称为 124M 参数模型；我们可以按如下方式核对该参数量：

In [4]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

Total number of parameters: 163,009,536


- 正如我们在上面所看到的，该模型拥有 163M 参数，而不是 124M；为什么会这样？
- 在原始 GPT-2 论文中，研究人员采用了权重绑定（weight tying），即复用 token 嵌入层（`tok_emb`）作为输出层，也就是令 `self.out_head.weight = self.tok_emb.weight`
- token 嵌入层将 50,257 维的独热编码输入 token 投影为 768 维的嵌入表示
- 输出层将 768 维的嵌入重新投影回 50,257 维表示，以便我们将其转换回词语（下一节将对此展开更详细说明）
- 因此，从权重矩阵的形状可以看出，嵌入层和输出层具有相同数量的权重参数
- 如果我们复用该权重矩阵，就可以显著节省参数量

In [5]:
print("Token embedding layer shape:", model.tok_emb.weight.shape)
print("Output layer shape:", model.out_head.weight.shape)

Token embedding layer shape: torch.Size([50257, 768])
Output layer shape: torch.Size([50257, 768])


- 在原始 GPT-2 论文中，研究人员复用了 token 嵌入矩阵作为输出矩阵
- 相应地，如果我们减去输出层的参数量，就能得到一个 124M 参数的模型：

In [6]:
total_params_gpt2 =  total_params - sum(p.numel() for p in model.out_head.parameters())
print(f"Number of trainable parameters considering weight tying: {total_params_gpt2:,}")

Number of trainable parameters considering weight tying: 124,412,160


- 在实践中，我发现在不使用权重绑定的情况下训练模型更容易，这也是我们在此处没有实现它的原因
- 不过，在第 5 章加载预训练权重时，我们将重新审视并应用这一权重绑定思想
- 最后，我们可以按如下方式计算模型的内存需求，这可以作为一个有用的参考基准：

In [7]:
# Calculate the total size in bytes (assuming float32, 4 bytes per parameter)
total_size_bytes = total_params * 4

# Convert to megabytes
total_size_mb = total_size_bytes / (1024 * 1024)

print(f"Total size of the model: {total_size_mb:.2f} MB")

Total size of the model: 621.83 MB


—练习：您可以尝试以下其他配置，参考[GPT-2试卷]（https://scholar.google.com/citations?view_op=view_citation&hl=en&user=dOad5HoAAAAJ&citation_for_view=dOad5HoAAAAJ:YsMSGLbcyi4C）。

- **GPT2-small**（我们已经实现的124M配置）
- "emb_dim" = 768
- "n_layers" = 12
- "n_heads" = 12

- * * GPT2-medium: * *
- "emb_dim" = 1024
- "n_layers" = 24
- "n_heads" = 16

- * * GPT2-large: * *
- "emb_dim" = 1280
- "n_layers" = 36
-“n_heads”= 20

- * * GPT2-XL: * *
- "emb_dim" = 1600
- n_layers = 48
- "n_heads" = 25